In [ ]:
import os
os.chdir("~/Treeshrew/pycistopic_260418")

In [ ]:
import pickle
import gzip 
from scipy import sparse
from scipy.io import mmwrite
import pycisTopic 
pycisTopic.__version__

import scanpy as sc

In [ ]:
#创建一个名为"outs"的目录
import os
out_dir = "outs"
os.makedirs(out_dir, exist_ok = True)

In [ ]:
cistopic_obj = pickle.load(open("~/Treeshrew/pycistopic_260418/cistopic_obj.pkl", "rb"))
print(cistopic_obj)

In [ ]:
models = pickle.load(open("~/Treeshrew/pycistopic_260418/outs/models.pkl", "rb"))
print(models)

In [ ]:
from pycisTopic.lda_models import evaluate_models
model = evaluate_models(
    models,
    metrics=['Minmo_2011'],
    #select_model = 220, #If not provided, the best model will be selected automatically based on the model quality metrics. Default: None
    return_model = False
)

In [ ]:
from pycisTopic.lda_models import evaluate_models
model = evaluate_models(
    models,
    metrics=['Minmo_2011','loglikelihood'],
    #select_model = 220, #If not provided, the best model will be selected automatically based on the model quality metrics. Default: None
    return_model = True
)

In [ ]:
from pycisTopic.lda_models import evaluate_models
model = evaluate_models(
    models,
    select_model = 200, #If not provided, the best model will be selected automatically based on the model quality metrics. Default: None
    return_model = True
)

In [ ]:
pickle.dump(
    model,
    open("outs/selected_model_200.pkl", "wb")
)

In [ ]:
cistopic_obj.add_LDA_model(model)#200

In [ ]:
pickle.dump(
    cistopic_obj,
    open("cistopic_obj.pkl", "wb"))

In [ ]:
cell_metadata=cistopic_obj.cell_data
cell_metadata

In [ ]:
cell_metadata['Subtype_ID_Num']=cell_metadata.index.str.split("_").str[0]
cell_metadata

In [ ]:
cell_metadata["Major_ID_Num"]=cell_metadata.index.str.split(".").str[0]
cell_metadata

In [ ]:
cistopic_obj.add_cell_data(cell_metadata)

In [ ]:
cistopic_obj.cell_data

In [ ]:
from pycisTopic.clust_vis import (
    find_clusters,
    run_umap,
    run_tsne,
    plot_metadata,
    plot_topic,
    cell_topic_heatmap
)

In [ ]:
cell_metadata=cistopic_obj.cell_data

In [ ]:
cell_metadata["Subtype_ID_Num"]
type(cell_metadata["Subtype_ID_Num"])

In [ ]:

Major_ID_Num_list = sorted(
    cell_metadata["Major_ID_Num"].unique().tolist(),
    key=lambda x: int(x.split(':')[0][1:])  # 提取"C"后面的数字部分
)
Major_ID_Num_list

In [ ]:
color_list=["#DDAD3B","#00BFFF","#9970AB","#00FF7F",## 1-4
                    "#FDCDAC","#3288BD","#32CD32","#48D1CC",## 5-8
                    "#FF33CC","#B15928","#6495ED","#D5004D",## 9-12
                    "#0000FF","#C2A5CF","#EE82EE","#FF7F00",## 13-16
                    "#9E0142","#B3CDE3","#87CEFA","#00FFFF",## 17-20
                    "#CDE8B9","#F4CAE4","#0099FF","#DE77AE"]## 21-24

In [ ]:
color_dict = {
    'Major_ID_Num': dict(zip(Major_ID_Num_list, color_list))
}

In [ ]:
cell_topic_heatmap(
    cistopic_obj,
    variables = ['Major_ID_Num'],
    scale = False,
    legend_loc_x = 1.25,
    legend_loc_y = -2.0,
    legend_dist_y = -2.4,
    figsize = (10, 10),
    color_dictionary=color_dict
)

In [ ]:
cell_topic_heatmap(
    cistopic_obj,
    variables = ['Major_ID_Num'],
    scale = True,
    legend_loc_x = 1.25,
    legend_loc_y = -2.0,
    legend_dist_y = -2.4,
    figsize = (10, 10),
    color_dictionary=color_dict
)

In [ ]:
import pandas as pd
import sklearn.preprocessing

# --- 1. 获取基础数据 (对应源码开头) ---
harmony = False # 如果你绘图时用了 harmony=True，请改为 True
model = cistopic_obj.selected_model
cell_topic = model.cell_topic_harmony if harmony else model.cell_topic
cell_data = cistopic_obj.cell_data
variables = ['Subtype_ID_Num',"Major_ID_Num"] # 替换为你绘图时用的变量

# --- 2. 标准化 (对应源码 scale 参数) ---
scale = False # 如果你绘图时 scale=True
if scale:
    cell_topic = pd.DataFrame(
        sklearn.preprocessing.StandardScaler().fit_transform(cell_topic),#Sklearn 的标准逻辑：StandardScaler 默认对矩阵的 每一列 (Column) 进行标准化
        index=cell_topic.index.to_list(),
        columns=cell_topic.columns,
    )

# --- 3. 还原源码的排序逻辑 ---
# A. 细胞排序 (按变量排序)
var = variables[0]
var_data = cell_data.loc[:, var].sort_values()
cell_topic_sorted_cells = cell_topic.loc[:, var_data.index.to_list()]

# B. 主题排序 (关键：源码中的 df.groupby(var).mean().idxmax()...)
# 先转置以便于计算
tmp_df = cell_topic_sorted_cells.transpose()
df_with_var = pd.concat([tmp_df, var_data], axis=1, sort=False)

# 寻找每个 Topic 均值最大的组，并排序得到 Topic 顺序
topic_order = df_with_var.groupby(var).mean().idxmax().sort_values().index.to_list()

# --- 4. 得到最终的热图矩阵 ---
# 此时行是 Topic，列是 Cell，顺序与官方热图一致
final_ordered_matrix = cell_topic_sorted_cells.loc[topic_order, :]

print("提取完成！矩阵维度：", final_ordered_matrix.shape)

In [ ]:
#细胞顺序 (从左到右)
final_ordered_matrix

In [ ]:
final_ordered_matrix.to_csv("TS_Topic_pseudocell.csv")

In [ ]:
ordered_cells = var_data.index.to_list()

In [ ]:
# 注意：如果 cluster_topics=True，顺序会被层次聚类覆盖
# 如果 cluster_topics=False（默认），顺序就是：
ordered_topics = topic_order

In [ ]:
# 1. 确保你的 final_ordered_matrix 已经提取出来
# 行是 Topic，列是 Cell Barcodes
# 我们先转置它，让 Cell 成为行，方便与 Metadata 对应
matrix_T = final_ordered_matrix.T

# 2. 将矩阵与细胞类型元数据合并
# 假设你的细胞类型存储在 cistopic_obj.cell_data['Seurat_cell_type']
cell_types = cistopic_obj.cell_data['Subtype_ID_Num']
merged_df = pd.concat([matrix_T, cell_types], axis=1)

# 3. 按细胞类型分组并计算平均值
# 这将得到一个：Index 为 Cell Type，Columns 为 Topic 的 DataFrame
avg_activity_df = merged_df.groupby('Subtype_ID_Num').mean()

# 4. 为了美观，再次转置回来（行是 Topic，列是 Cell Type）
avg_activity_df = avg_activity_df.T

print(avg_activity_df)

In [ ]:
avg_activity_df.to_csv("TS_Average_Topic_Subtype_ID_Num.csv")

In [ ]:
cell_topic_heatmap(
    cistopic_obj,
    variables = ['Subtype_ID_Num',"Major_ID_Num"],
    scale = False,
    legend_loc_x = 1.25,
    legend_loc_y = -5.0,
    legend_dist_y = -5.4,
    figsize = (10, 10),
    color_dictionary=color_dict,
    save="~/Treeshrew/pycistopic_260418/cell_topic_heatmap_Major_subtype_ID_Num.pdf"
)

In [ ]:
color_dict["Subtype_ID_Num"]

In [ ]:
run_tsne(
    cistopic_obj,
    target  = 'region', scale=True)

In [ ]:
plot_metadata(
    cistopic_obj,
    reduction_name='tSNE',
    variables=['cisTopic_log_nr_frag','cisTopic_log_nr_acc' ],
    target='region', num_columns=2,
    text_size=10,
    dot_size=0.003,
    alpha=0.5)

In [ ]:
pickle.dump(
    cistopic_obj,
    open("cistopic_obj.pkl", "wb")
)

In [ ]:
from pycisTopic.topic_binarization import binarize_topics

In [ ]:
help(binarize_topics)

In [ ]:
region_bin_topics_top_3k = binarize_topics(
    cell_or_region_topic_prob=cistopic_obj.selected_model.region_topic,
    cell_or_region_names=cistopic_obj.region_names, 
    method='ntop',
    smooth_topics=True,
    ntop = 3_000
)

In [ ]:
region_bin_topics_otsu = binarize_topics(
    cell_or_region_topic_prob=cistopic_obj.selected_model.region_topic,
    cell_or_region_names=cistopic_obj.region_names,
    method='otsu',
    smooth_topics=True,
    nbins=100
)

In [ ]:
type(region_bin_topics_otsu)

In [ ]:
region_bin_topics_otsu[0][0]

In [ ]:
type(region_bin_topics_top_3k)

In [ ]:
region_bin_topics_top_3k[0][0]

In [ ]:
os.makedirs(os.path.join(out_dir, "region_sets"), exist_ok=True)
os.makedirs(os.path.join(out_dir, "region_sets","Topics_otsu"), exist_ok=True)
# 保存每个topic的region
from pycisTopic.utils import region_names_to_coordinates

# 假设第一个列表包含所有topic的region信息
topics_regions = region_bin_topics_otsu[0]

for topic_idx, topic_regions in enumerate(topics_regions):
    # 将region名称转换为坐标并排序
    region_coords = region_names_to_coordinates(topic_regions).sort_values(
        ["Chromosome", "Start", "End"]
    )
    region_coords["Start"]=region_coords["Start"]-1 #0-based
    # 保存为BED文件，使用更有意义的文件名
    filename = f"topic_{topic_idx + 1}.bed"  # 从1开始计数
    region_coords.to_csv(
        os.path.join(out_dir, "region_sets", "Topics_otsu", filename),
        sep="\t",
        header=False, 
        index=False
    )

print(f"成功保存了 {len(topics_regions)} 个topic的region文件")

In [ ]:
os.makedirs(os.path.join(out_dir, "region_sets"), exist_ok=True)
os.makedirs(os.path.join(out_dir, "region_sets","Topics_top3k"), exist_ok=True)
# 保存每个topic的region
from pycisTopic.utils import region_names_to_coordinates

# 假设第一个列表包含所有topic的region信息
topics_regions = region_bin_topics_top_3k[0]

for topic_idx, topic_regions in enumerate(topics_regions):
    # 将region名称转换为坐标并排序
    region_coords = region_names_to_coordinates(topic_regions).sort_values(
        ["Chromosome", "Start", "End"]
    )
    region_coords["Start"]=region_coords["Start"]-1 #0-based
    # 保存为BED文件，使用更有意义的文件名
    filename = f"topic_{topic_idx + 1}.bed"  # 从1开始计数
    region_coords.to_csv(
        os.path.join(out_dir, "region_sets", "Topics_top3k", filename),
        sep="\t",
        header=False, 
        index=False
    )

print(f"成功保存了 {len(topics_regions)} 个topic的region文件")

In [ ]:
topics_regions
region_coords = region_names_to_coordinates(topic_regions[0]).sort_values(
        ["Chromosome", "Start", "End"]
    )
region_coords   

In [ ]:

region_final_list = list({item for sublist in region_bin_topics_otsu[0] for item in sublist})
len(region_final_list)


In [ ]:
# 解包数据
topics_regions_list, topics_scores_list, topics_thresholds = region_bin_topics_otsu

# 为每个region分配概率最高的topic
region_primary_topic = {}

# 遍历所有topic
for topic_idx in range(len(topics_regions_list)):
    regions = topics_regions_list[topic_idx]
    scores = topics_scores_list[topic_idx]
    
    # 遍历该topic中的所有region
    for region_idx, region_name in enumerate(regions):
        score = scores[region_idx]
        
        # 如果region不在字典中，或者当前topic的分数更高，则更新
        if region_name not in region_primary_topic or score > region_primary_topic[region_name]['score']:
            region_primary_topic[region_name] = {
                'topic': f"Topic{topic_idx+1}", #发现应该index+1以确保topic名称从1开始计
                'score': score
            }


In [ ]:
import pandas as pd

In [ ]:
# 一行代码转换
region_topic_list = [[region, info['topic'], info['score']] for region, info in region_primary_topic.items()]

# 创建DataFrame
region_metadata = pd.DataFrame(region_topic_list, columns=['region', 'assigned_topic', 'max_probability'])
region_metadata = region_metadata.set_index('region')

print(region_metadata.head())

In [ ]:
region_metadata.to_csv("~/Treeshrew/pycistopic_260418/outs/region_sets/Topics_otsu/region_metadata.csv")

In [ ]:
cistopic_obj.add_region_data(region_metadata)
cistopic_obj.region_data

In [ ]:
pickle.dump(
    cistopic_obj,
    open("cistopic_obj.pkl", "wb")
)

In [ ]:
cistopic_obj.region_data.to_csv("TS_region_data.csv")

In [ ]:
plot_metadata(
    cistopic_obj,
    reduction_name='tSNE',
    variables=['assigned_topic'],
    target='region', num_columns=2,
    text_size=5,
    dot_size=0.003,
    alpha=0.5)

In [ ]:
region_final_list=region_metadata.index.to_list()
region_final_list

In [ ]:
len(region_final_list)

In [ ]:
from pycisTopic.clust_vis import (
    find_clusters,
    run_umap,
    run_tsne,
    plot_metadata,
    plot_topic,
    cell_topic_heatmap
)

In [ ]:
run_tsne(
    cistopic_obj,
    target  = 'region', scale=True,
    selected_features=region_final_list)

In [ ]:
plot_metadata(
    cistopic_obj,
    reduction_name='tSNE',
    variables=['cisTopic_log_nr_frag','cisTopic_log_nr_acc' ],
    target='region', num_columns=2,
    text_size=10,
    dot_size=0.003,
    alpha=0.5,
    selected_features=region_final_list)

In [ ]:
plot_topic(
    cistopic_obj,
    reduction_name = 'tSNE',
    target = 'region',
    num_columns=5
)

In [ ]:
plot_metadata(
    cistopic_obj,
    reduction_name='tSNE',
    variables=['assigned_topic' ],
    target='region', num_columns=2,
    text_size=5,
    dot_size=0.003,
    alpha=0.5,
    selected_features=region_final_list)

In [ ]:
pickle.dump(
    cistopic_obj,
    open("cistopic_obj.pkl", "wb"))

In [ ]:
cistopic_obj.projections['region']['tSNE'].to_csv("TS_region_binarized_scaled_tsne.csv")